<a href="https://colab.research.google.com/github/marcochen819/ETF-SUMMARY/blob/main/0421-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

def get_data(date_str):
    url = f"https://www.twse.com.tw/rwd/zh/fund/T86?response=json&date={date_str}&selectType=0099P"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        data = requests.get(url, headers=headers).json()
        if data.get("stat") == "OK":
            df = pd.DataFrame(data["data"], columns=data["fields"])
            # 只拿要的那五個欄位
            cols = ["證券代號", "證券名稱", "外陸資買賣超股數(不含外資自營商)",
                    "投信買賣超股數", "自營商買賣超股數", "三大法人買賣超股數"]
            df = df[cols]
            df.insert(0, '日期', date_str)
            return df
    except:
        pass
    return None

# 自動產生日期 (例如抓一整個月)
start_date = datetime(2026, 1, 1)
end_date = datetime(2026, 1, 14)
all_dfs = []

current = start_date
while current <= end_date:
    d_str = current.strftime("%Y%m%d")

    # 週末直接跳過，省下請求時間
    if current.weekday() < 5:
        print(f"檢查 {d_str}...", end="\r")
        res = get_data(d_str)
        if res is not None:
            all_dfs.append(res)
            time.sleep(3) # 有抓到資料才停頓

    current += timedelta(days=1)

# 合併結果
if all_dfs:
    pd.concat(all_dfs).to_csv("ETF_Analysis.csv", encoding="utf-8", index=False)
    print("\n抓取完成！")


抓取完成！
